В данном ноутбуке мы будем получать через api superjob 

In [2]:
%pip install python-dotenv requests


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import dotenv
import os

dotenv.load_dotenv(dotenv_path='../../.env')

SUPERJOB_SECRET = os.getenv('SUPERJOB_SECRET_KEY')
SUPERJOB_CLIENT_ID = os.getenv('SUPERJOB_CLIENT_ID')
SUPERJOB_LOGIN = os.getenv('SUPERJOB_LOGIN')
SUPERJOB_PASSWORD = os.getenv('SUPERJOB_PASSWORD')

In [6]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging

In [7]:
# Авторизуемся

params = {
    'client_id': SUPERJOB_CLIENT_ID,
    'client_secret': SUPERJOB_SECRET,
    'login': SUPERJOB_LOGIN,
    'password': SUPERJOB_PASSWORD,
}

auth_response = requests.post('https://api.superjob.ru/2.0/oauth2/password/', params=params)

In [8]:
token = auth_response.json()['access_token']
headers = {
    "Authorization": f"Bearer {token}",
    "X-Api-App-Id": SUPERJOB_SECRET
}

In [9]:
industry = requests.get('https://api.superjob.ru/2.0/catalogues/', headers=headers)
industry_res = industry.json()
industry_res[0]

{'title_rus': 'IT, Интернет, связь, телеком',
 'url_rus': 'it-internet-svyaz-telekom',
 'title': 'IT, Интернет, связь, телеком',
 'title_trimmed': 'IT, Интернет, связь,...',
 'key': 33,
 'positions': [{'title_rus': 'AI',
   'url_rus': 'ai',
   'title': 'AI',
   'id_parent': 33,
   'key': 651},
  {'title_rus': 'CRM-системы',
   'url_rus': 'crm-sistemy',
   'title': 'CRM-системы',
   'id_parent': 33,
   'key': 603},
  {'title_rus': 'Data Science',
   'url_rus': 'data-science',
   'title': 'Data Science',
   'id_parent': 33,
   'key': 627},
  {'title_rus': 'DevOps',
   'url_rus': 'devops',
   'title': 'DevOps',
   'id_parent': 33,
   'key': 628},
  {'title_rus': 'SRE',
   'url_rus': 'sre',
   'title': 'SRE',
   'id_parent': 33,
   'key': 629},
  {'title_rus': 'Web-верстка',
   'url_rus': 'web-verstka',
   'title': 'Web-верстка',
   'id_parent': 33,
   'key': 36},
  {'title_rus': 'Администрирование баз данных',
   'url_rus': 'administrirovanie-baz-dannyh',
   'title': 'Администрирование ба

Здесь сразу можно отметить, что 'IT, Интернет, связь, телеком' - первая отрасль и  ее 'key': 33

In [10]:
titles = []
for catalogue in industry_res:
    titles.append(catalogue['title_rus'])
print(titles)

['IT, Интернет, связь, телеком', 'Административная работа, секретариат, АХО', 'Банки, инвестиции, лизинг', 'Безопасность, службы охраны', 'Бухгалтерия, финансы, аудит', 'Государственная служба', 'Дизайн', 'Домашний персонал', 'Закупки, снабжение', 'Искусство, культура, развлечения', 'Кадры, управление персоналом', 'Консалтинг, стратегическое развитие', 'Маркетинг, реклама, PR', 'Медицина, фармацевтика, ветеринария', 'Наука, образование, повышение квалификации', 'Некоммерческие организации, волонтерство', 'Обмен персоналом', 'Продажи', 'Промышленность, производство', 'Рабочий персонал', 'Сельское хозяйство', 'Службы доставки', 'СМИ, издательства', 'Спорт, фитнес, салоны красоты, SPA', 'Страхование', 'Строительство, проектирование, недвижимость', 'Сырье', 'Топ-персонал', 'Транспорт, логистика, ВЭД', 'Туризм, гостиницы, общественное питание', 'Услуги, ремонт, сервисное обслуживание', 'Юриспруденция']


Список отраслей: ['IT, Интернет, связь, телеком', 'Административная работа, секретариат, АХО', 'Банки, инвестиции, лизинг', 'Безопасность, службы охраны', 'Бухгалтерия, финансы, аудит', 'Государственная служба', 'Дизайн', 'Домашний персонал', 'Закупки, снабжение', 'Искусство, культура, развлечения', 'Кадры, управление персоналом', 'Консалтинг, стратегическое развитие', 'Маркетинг, реклама, PR', 'Медицина, фармацевтика, ветеринария', 'Наука, образование, повышение квалификации', 'Некоммерческие организации, волонтерство', 'Обмен персоналом', 'Продажи', 'Промышленность, производство', 'Рабочий персонал', 'Сельское хозяйство', 'Службы доставки', 'СМИ, издательства', 'Спорт, фитнес, салоны красоты, SPA', 'Страхование', 'Строительство, проектирование, недвижимость', 'Сырье', 'Топ-персонал', 'Транспорт, логистика, ВЭД', 'Туризм, гостиницы, общественное питание', 'Услуги, ремонт, сервисное обслуживание', 'Юриспруденция']

Тк мы рассматриваем сферу  it, то нам подойдет отрасль 'IT, Интернет, связь, телеком'. Однако также есть отрасль 'Дизайн'. UX/UI дизайнеры и исследователи тоже относятся к ИТ-профессиям. Надо проверить, возможно, эти ИТ-направления относятся к данной отрасли, а не к 'IT, Интернет, связь, телеком'.


In [11]:
titles_in_design = []
for position in  industry_res[6]['positions']:
    titles_in_design.append(position['title_rus'])
print(titles_in_design)

['Web-дизайн (UI/UX)', 'Архитектура', 'Аудио / Саунд-дизайн', 'Верстка и полиграфия', 'Видео, CGI, анимация', 'Графический дизайн', 'Дизайн интерьера', 'Иллюстрации', 'Ландшафтный дизайн', 'Мода\xa0и\xa0Аксессуары', 'Промышленный дизайн', 'Рекламный дизайн / Ивенты, инсталляции, стенды', 'Фотография', 'Другое', 'Начало карьеры, мало опыта']


И правда - из всех направлений дизайна также подходит направление 'Web-дизайн (UI/UX)' как часть IT-сферы.

In [12]:
industry_res[6]['positions'][0]

{'title_rus': 'Web-дизайн (UI/UX)',
 'url_rus': 'web-dizajn',
 'title': 'Web-дизайн (UI/UX)',
 'id_parent': 62,
 'key': 35}

In [13]:
# Получаем список вакансий по веб-дизигну с помощью key=35

params = {
    'catalogues': 35,
}

vacs_design = requests.get('https://api.superjob.ru/2.0/vacancies/', headers=headers, params=params)
design_res = vacs_design.json()
design_res['total']

43

по 40 вакансий на page => 2 pages надо пройти чтобы скачать всё в один датафрейм

In [14]:
res_vacs_design = []
params = {
    'catalogues': 35,
    'count': 40,
    'page': 0
}

for page in range(2):
    params['page']=page
    vacs_design = requests.get('https://api.superjob.ru/2.0/vacancies/', headers=headers, params=params)
    vacs_design_json = vacs_design.json()
    res_vacs_design.extend(vacs_design_json['objects'])
    
df_design_vacs=pd.DataFrame(res_vacs_design)
df_design_vacs.head(3)

,contacts_hidden,contact_hiding_cause,canEdit,is_closed,id,id_client,payment_from,payment_to,date_pub_to,date_archived,...,highlight,age_from,age_to,gender,firm_name,firm_activity,link,isBlacklisted,latitude,longitude
0,True,no_resume,False,False,50780033,14215,31500,32000,1775822012,1771476907,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Почта России,Почта России сейчас — это важная часть социаль...,https://bizhbulyak.superjob.ru/vakansii/operat...,False,53.698341,54.268978
1,True,no_resume,False,False,51646102,4944790,0,0,1777646146,0,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Лаборатория Касперского,Обязанности:Требования:Условия.,https://www.superjob.ru/vakansii/ux-51646102.html,False,NaN,NaN
2,True,no_resume,False,False,51697899,11714,92000,105000,1777802103,0,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://www.superjob.ru/vakansii/ux-issledovat...,False,55.764221,37.617321


In [15]:
key_it = 33

res_vacs_it = []
params = {
    'catalogues': key_it,
    'count': 40,
    'page': 0
}
while True:
    
    vacs_it = requests.get('https://api.superjob.ru/2.0/vacancies/', headers=headers, params=params)
    if vacs_it.status_code != 200:
        break
    vacs_it_json = vacs_it.json()

    if not vacs_it_json['objects']:
        break
    res_vacs_it.extend(vacs_it_json['objects'])
    params['page'] += 1
    

(Источник, где я смотрела способ сбора данных со всех страниц: https://sky.pro/wiki/media/kak-ispolzovat-python-dlya-raboty-s-rest-api/)

In [16]:
df_it = pd.DataFrame(res_vacs_it)
df_it.head(3)

,contacts_hidden,contact_hiding_cause,canEdit,is_closed,id,id_client,payment_from,payment_to,date_pub_to,date_archived,...,age_from,age_to,gender,firm_name,firm_activity,link,isBlacklisted,latitude,longitude,video
0,True,no_resume,False,False,51697590,11714,0,0,1777812604,1772612101,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://ufa.superjob.ru/vakansii/dezhurnyj-spe...,False,54.783783,56.122543,NaN
1,True,no_resume,False,False,51697719,11714,0,36540,1777799110,0,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://furmanov.superjob.ru/vakansii/sistemny...,False,57.257141,41.117516,NaN
2,True,no_resume,False,False,51697696,11714,50000,60000,1777798205,0,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://www.superjob.ru/vakansii/sistemnyj-adm...,False,46.114944,32.907536,NaN


Так как выбранная отрасль - 'IT, Интернет, связь, телеком' содержит не только ИТ сферу, но и работу со связью и телекомом, то стоит подробнее рассмотреть, что необходимо исключить из выборки.

In [17]:
titles_in_it_industry = []
for position in  industry_res[0]['positions']:
    titles_in_it_industry.append(position['title_rus'])
print(titles_in_it_industry)

['AI', 'CRM-системы', 'Data Science', 'DevOps', 'SRE', 'Web-верстка', 'Администрирование баз данных', 'Аналитика', 'Внедрение и сопровождение ПО', 'Игровое ПО / Геймдев', 'Инжиниринг', 'Интернет, создание и поддержка сайтов', 'Информационная безопасность', 'Киберспорт', 'Компьютерная анимация\xa0и\xa0мультимедиа', 'Контент', 'Мобильная разработка', 'Нейросети / Искусственный интеллект', 'Оптимизация, SEO', 'Передача данных и доступ в интернет', 'Разработка и сопровождение банковского ПО', 'Разработка, программирование', 'Сетевые технологии', 'Системная интеграция', 'Системное администрирование', 'Системы автоматизированного проектирования (САПР)', 'Системы управления предприятием (ERP)', 'Сотовые, беспроводные технологии', 'Телекоммуникации и связь', 'Тестирование, QA', 'Техническая документация', 'Техническая поддержка', 'Управление продуктом', 'Управление проектами', 'Управление разработкой', 'Юзабилити', 'Другое', 'Начало карьеры, мало опыта']


Список специализаций отрасли: ['AI', 'CRM-системы', 'Data Science', 'DevOps', 'SRE', 'Web-верстка', 'Администрирование баз данных', 'Аналитика', 'Внедрение и сопровождение ПО', 'Игровое ПО / Геймдев', 'Инжиниринг', 'Интернет, создание и поддержка сайтов', 'Информационная безопасность', 'Киберспорт', 'Компьютерная анимация\xa0и\xa0мультимедиа', 'Контент', 'Мобильная разработка', 'Нейросети / Искусственный интеллект', 'Оптимизация, SEO', 'Передача данных и доступ в интернет', 'Разработка и сопровождение банковского ПО', 'Разработка, программирование', 'Сетевые технологии', 'Системная интеграция', 'Системное администрирование', 'Системы автоматизированного проектирования (САПР)', 'Системы управления предприятием (ERP)', 'Сотовые, беспроводные технологии', 'Телекоммуникации и связь', 'Тестирование, QA', 'Техническая документация', 'Техническая поддержка', 'Управление продуктом', 'Управление проектами', 'Управление разработкой', 'Юзабилити', 'Другое', 'Начало карьеры, мало опыта'].
Здесь нам точно не подойдут: 'Телекоммуникации и связь', 'Техническая поддержка'

Уберем из датафрейма вакансии, которые относятся к этим специализациям

In [18]:
def del_vacs(catalogues, list_specialization):
    for cat in catalogues:
        for pos in cat['positions']:
            if pos['title'] in list_specialization:
                return True
    return False
            
list_specialization = ['Телекоммуникации и связь', 'Телекоммуникации и связь']
mask_del = df_it['catalogues'].apply(lambda x: del_vacs(x, list_specialization))

df_it_cleaned = df_it[~mask_del]
len(df_it_cleaned)

472

После удаления неподходящих специализаций осталось 466 строк.

Соединим датафрейм с веб-дизайнерами с датафреймом вакансий из отрасли 'IT, Интернет, связь, телеком' 

In [19]:
all_vacs_api = pd.concat([df_it_cleaned, df_design_vacs], axis=0, ignore_index=True)
all_vacs_api.head(3)

,contacts_hidden,contact_hiding_cause,canEdit,is_closed,id,id_client,payment_from,payment_to,date_pub_to,date_archived,...,age_from,age_to,gender,firm_name,firm_activity,link,isBlacklisted,latitude,longitude,video
0,True,no_resume,False,False,51697590,11714,0,0,1777812604,1772612101,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://ufa.superjob.ru/vakansii/dezhurnyj-spe...,False,54.783783,56.122543,NaN
1,True,no_resume,False,False,51697719,11714,0,36540,1777799110,0,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://furmanov.superjob.ru/vakansii/sistemny...,False,57.257141,41.117516,NaN
2,True,no_resume,False,False,51697696,11714,50000,60000,1777798205,0,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://www.superjob.ru/vakansii/sistemnyj-adm...,False,46.114944,32.907536,NaN


Изучив документацию, мы выяснили, что некоторые признаки нет смысла анализировать насколько они нам подходят, проверять пропуски и выбросы, так как они не дадут нам полезную информацию. Например, признаки относящиеся к конкретному пользователю (относительно которого и собираются данные), в контексте определения полезных закономерностей для компании, выпускающей курсы, такие значения несут крайне мало информации. Также малоинформативно точное расположение компании, контакты сотрудника, разместившего вакансию, и признаки, показывающие уровень возможного взаимодействия с ней. И еще несколько столбцов, которые можно отбросить, рассмотрев только их описание.
Удалим эти признаки для более удобной дальнейшей работы. Те, которые требуют более детального анализа пока оставим и рассмотрим позже.

In [20]:
df_api_clean = all_vacs_api.drop(columns=['contacts_hidden', 'contact_hiding_cause', 'canEdit', 'date_pub_to', 'address', 'metro', 'covid_vaccination_requirement', 'external_url', 'anonymous', 'maritalstatus', 'children', 'driving_licence', 'already_sent_on_vacancy', 'rejected', 'phone', 'favorite', 'client_logo', 'latitude', 'longitude', 'video', 'isBlacklisted', 'is_closed', 'highlight', 'response_info', 'trud_vsem', 'link'])

Признак 'town' оформлен виде словарей, пример:

{'id': 173,
 'title': 'Уфа',
 'declension': 'в Уфе',
 'hasMetro': False,
 'genitive': 'Уфы'}

 
 для более удобной дальнейшей работы разобьейм на 2 столбца: название города (для более удобного анализа) и id (для дальнейшего получаения области по нему)

In [21]:
df_api_clean['town_name'] = df_api_clean['town'].apply(lambda x: x['title'])
df_api_clean['town'] = df_api_clean['town'].apply(lambda x: x['id'])

Получим с помощью апи таблицу с городами, чтобы в дальнейшем получить регионы(области), где они находятся

In [22]:
params = {
    'all': 1
}
towns = requests.get('https://api.superjob.ru/2.0/towns/', headers=headers, params=params)
res_towns = towns.json()
df_towns = pd.DataFrame(res_towns['objects']).iloc[:, :3]
len(res_towns['objects'])

5285

In [223]:
to_csv_towns = pd.DataFrame(res_towns['objects'])

In [23]:
df_towns.head(3)

,id,id_region,id_country
0,4,46,1
1,14,41,1
2,13,50,1


Теперь по ID города к итоговому датафрейму присоединим ID региона и страны

In [ ]:
df_vacs_towns = df_api_clean.merge(df_towns, left_on='town', right_on='id', how='left')
df_vacs_towns = df_vacs_towns.drop(columns='id_y')
df_vacs_towns = df_vacs_towns.rename(columns={'id_x': 'id'})
df_vacs_towns.head(3)

,id,id_client,payment_from,payment_to,date_archived,date_published,profession,work,compensation,candidat,...,agency,town,age_from,age_to,gender,firm_name,firm_activity,town_name,id_region,id_country
0,51697590,11714,0,0,1772612101,1775739004,Дежурный специалист по информационной безопасн...,None,None,"ФФКУ ""Налог-Сервис"" ФНС России по ЦОД- это про...",...,"{'id': 1, 'title': 'прямой работодатель'}",173,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",Уфа,80,1
1,51697719,11714,0,36540,0,1775725510,Системный администратор,None,None,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",...,"{'id': 1, 'title': 'прямой работодатель'}",955,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",Фурманов,24,1
2,51697696,11714,50000,60000,0,1775724605,Системный администратор,None,None,ФКУ Налог-Сервис ФНС России - это продвинутая ...,...,"{'id': 1, 'title': 'прямой работодатель'}",2267,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",Скадовск,122,1


Теперь с помощью апи получим таблицы с регионами и странами

In [25]:
params = {
    'all': 1
}
regions = requests.get('https://api.superjob.ru/2.0/regions/', headers=headers, params=params)
res_regions = regions.json()
df_regions = pd.DataFrame(res_regions['objects']).loc[:,['id', 'title']]
len(res_regions['objects'])

104

In [26]:
countries = requests.get('	https://api.superjob.ru/2.0/countries/', headers=headers)
res_countries = countries.json()
df_countries = pd.DataFrame(res_countries['objects'])
len(res_countries['objects'])

18

In [76]:
df_countries = df_countries.rename(columns={'id':'id_country','title': 'country_title'})
df_regions = df_regions.rename(columns={'id':'id_region', 'title': 'region__title'})

Соединим названия регионов и стран с общим датафреймом

In [153]:
df_vacs_regions = df_vacs_towns.merge(df_regions, on='id_region', how='left')
df_api_res = df_vacs_regions.merge(df_countries, on='id_country', how='left')

In [154]:
df_api_res.head(3)


,id,id_client,payment_from,payment_to,date_archived,date_published,profession,work,compensation,candidat,...,age_from,age_to,gender,firm_name,firm_activity,town_name,id_region,id_country,region__title,country_title
0,51697590,11714,0,0,1772612101,1775739004,Дежурный специалист по информационной безопасн...,None,None,"ФФКУ ""Налог-Сервис"" ФНС России по ЦОД- это про...",...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",Уфа,80,1,Республика Башкортостан,Россия
1,51697719,11714,0,36540,0,1775725510,Системный администратор,None,None,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",Фурманов,24,1,Ивановская область,Россия
2,51697696,11714,50000,60000,0,1775724605,Системный администратор,None,None,ФКУ Налог-Сервис ФНС России - это продвинутая ...,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",Скадовск,122,1,Херсонская область,Россия


ОБРАБОТКА ДУБЛИКАТОВ

Проверим получившийся датафрейм на наличие полных дубликатов

In [155]:
df_api_res['id'].duplicated().sum()

10

In [156]:
df_api_res[df_api_res['id'].duplicated(keep=False)]['catalogues'][18]

[{'id': 62,
  'title': 'Дизайн',
  'key': 62,
  'positions': [{'id': 35, 'title': 'Web-дизайн (UI/UX)', 'key': 35}]},
 {'id': 33,
  'title': 'IT, Интернет, связь, телеком',
  'key': 33,
  'positions': [{'id': 38, 'title': 'Аналитика', 'key': 38},
   {'id': 48, 'title': 'Разработка, программирование', 'key': 48},
   {'id': 59, 'title': 'Юзабилити', 'key': 59},
   {'id': 613, 'title': 'Управление продуктом', 'key': 613}]}]

Так 'catalogues' каждой вакансии состоит из нескольких отраслей и специализаций, нашлось 11 таких, которые отнесены и к отрасли 'IT, Интернет, связь, телеком', и к специализации 'Web-дизайн (UI/UX)', при слиянии двух датафреймов 11 вакансий задублировались, но остальная информация у них идентична, поэтому их можно просто удалить.

In [157]:
mask_dupl = df_api_res['id'].duplicated()
df_api_res = df_api_res[~mask_dupl]

FEATURE ENGINEERING

Feature extraction

Еще ранее было замечено, что некоторые признаки записаны в виде не строкового или числового значения, а словарей. Для удобства дальнейшего анализа обработаем их.

Всего в текущем датафрейме 8 таких признаков: type_of_work, place_of_work, education, experience, client, agency, gender, catalogues. Из них 6 (кроме client и catalogues) состоят из ID и title, пример:

{'id': 6, 'title': 'Полный рабочий день'},

но для дальнейшего анализа нам достаточно просто названий, поэтому соответсвующе обработаем эти признаки

In [158]:
df_api_res.loc[:, 'type_of_work'] = df_api_res['type_of_work'].apply(lambda x: x['title'])
df_api_res.loc[:, 'place_of_work'] = df_api_res['place_of_work'].apply(lambda x: x['title'])
df_api_res.loc[:, 'education'] = df_api_res['education'].apply(lambda x: x['title'])
df_api_res.loc[:, 'experience'] = df_api_res['experience'].apply(lambda x: x['title'])
df_api_res.loc[:, 'agency'] = df_api_res['agency'].apply(lambda x: x['title'])
df_api_res.loc[:, 'gender'] = df_api_res['gender'].apply(lambda x: x['title'])

Теперь рассмотрим client - компании, которые выкладывают вакансии

In [159]:
df_api_res.loc[70, 'client']

{'id': 15563,
 'title': 'Дом Моды HENDERSON',
 'link': 'https://www.superjob.ru/clients/dom-mody-henderson-15563/vacancies.html',
 'industry': [],
 'description': 'Дом моды HENDERSON более 20 лет предлагает мужчинам элегантную и стильную одежду для работы и отдыха. Сегодня HENDERSON присутствует более чем в 150 крупнейших торговых центрах, расположенных более чем в 50 городах России.\n\nКорпоративная культура HENDERSON основана на сложившейся системе ценностей: вере, уважении, сотрудничестве, знании и ответственности. Мы рады предложить нашим сотрудникам благоприятные условия труда на основе взаимного уважения, возможность внести свой вклад в дальнейший успех нашей компании, а также уникальные преимущества и широкие возможности для личного и профессионального роста.\n\nГК HENDERSON-Россия относится с уважением к своим ценностям и с уверенностью смотрит в будущее. В нашей компании всегда найдется место для умных, целеустремленных, ответственных сотрудников, умеющих работать в команде.\n

Из полезной информации, которую можно попробовать использовать в нашем анализе можно оставить: 'staff_count'. Остальное можно убрать. Название уже есть как признак - 'firm_title', описание (description) - выделен в столбце 'firm_activity'. Также изначельно заинтересовал признак - 'vacancy_count', в нашем понимании это количество вакансий в компании, размещенные на SuperJob, по этим данным можно было бы определить потребность в сотрудниках в разных компаниях/отраслях, но в документации не указано, каким именно образом получаются эти значения, а наше предположение о количестве не подтвердились, поэтому было принято решение не использовать данные, которые мы не можем интерпретировать.

Было бы полезно воспользоваться признаком 'industry', но это необязательный признак для заполнения, поэтому у большинства компаний он пуст, и не стоит его рассматривать.

In [160]:
df_api_res.loc[:, 'client_staff_count'] = df_api_res['client'].apply(lambda x: x.get('staff_count'))

In [161]:
df_api_res = df_api_res.drop(columns='client')

И осталось обработать 'catalogues' (список категорий и отраслей вакансии).
Изначально отбирались вакансии из двух отраслей: 
  'title': 'IT, Интернет, связь, телеком' и 'Дизайн', но так как на SuperJob для одной вакансии можно подобрать несколько отраслей и категорий в них(positions), то необходимо это отдельно обработать, оставив только то, что связано с ИТ, чтобы данные соответсвовали цели проекта. Для более удобного дальнейшего анализа категории разместим в отдельный столбец,оформим в виде списка.

In [162]:
def get_position(list_cat):
    res_pos = []
    for cat in list_cat:
        if cat['key'] == 33:
            for pos in cat['positions']:
                res_pos.append(pos['title'])
        if cat['key'] == 62:
            for pos in cat['positions']:
                if pos['title'] == 'Web-дизайн (UI/UX)':
                    res_pos.append(pos['title'])
    return res_pos

df_api_res['position'] = df_api_res['catalogues'].apply(get_position)

In [163]:
df_api_res = df_api_res.drop(columns='catalogues')

Также рассмотрим два признака с зарплатой (значения за 'от' и 'до'). ЗП приводим к одному значению для удобства анализа. Таким образом, где есть вилка от-до - берем среднее; где только одно значение (от или до) - оставляем его.


In [199]:
def salary(payment_from, payment_to):
    if payment_from>0 and payment_to>0:
        return (payment_from + payment_to) / 2
    elif payment_from>0:
        return payment_from
    return payment_to

df_api_res['salary'] = df_api_res[['payment_from', 'payment_to']].apply(lambda x: salary(x['payment_from'], x['payment_to']), axis = 1)

In [201]:
df_api_res = df_api_res.drop(columns=['payment_from','payment_to'])

Feature selection

Ранее мы убрали явно не подходящие признаки, теперь после дополнительного анализа признаков, мы определили еще несколько, которые стоит удалить. 

Для начала можно убрать все признаки, которые являются ID, так как необходимые соединения мы уже проделали, а для анализа будет более наглядно использование названий компаний/вакансий/городов и тд (id, id_client, id_region, id_country, town).

In [164]:
df_api_res = df_api_res.drop(columns=['id', 'id_client', 'id_region', 'id_country', 'town'])

Архивные данные нам недоступны исходя из условий, указанных в документации (вакансии, находящиеся в архиве (*только если была совершена авторизация под пользователем-работодателем)), поэтому можно удалить следубщие признаки: date_archived, is_archive, is_storage.

Также для анализа информации для выявления инсайтов для компании, выпускающей курсы, мы посчитали менее информативными следующие признаки: date_published, moveable, agreement, agency.

In [165]:
df_api_res = df_api_res.drop(columns=[ 'date_archived', 'is_archive', 'is_storage',  'date_published', 'moveable', 'agreement', 'agency'])

И некоторые признаки заполнены одинаковой информацией/в большинстве вакансий отсутсвуют (вероятно это не самые важные критерии для работадателей, поэтому они их не указывают, а эти признаки как раз опциональные), следовательно для анализа они мало информативны: languages, age_from, age_to, gender

In [166]:
df_api_res['age_from'].unique()

array([0])

In [167]:
df_api_res['age_to'].unique()

array([0])

In [168]:
df_api_res['gender'].unique()

array(['Не имеет значения'], dtype=object)

In [169]:
df_api_res['languages'].apply(lambda x: len(x)).sum()

24

In [170]:
df_api_res = df_api_res.drop(columns=['languages', 'age_from', 'age_to', 'gender'])

Также мы отдельно рассмотрели признаки: candidat, firm_activity и vacancyRichText, и выяснили, что candidat полностью дублирует vacancyRichText, но упуская оформление текста, а firm_activity во многих вакансиях заполнен некорректно, поэтому мы приняли решение оставить только vacancyRichText.

In [171]:
df_api_res['candidat'][0]

'ФФКУ "Налог-Сервис" ФНС России по ЦОД- это продвинутая IT - организация с новейшим оборудованием и современными технологиями. Мы занимаемся сопровождением IT - инфраструктуры центров обработки данных (ЦОД) налоговой службы.\nНаша организация - крупнейшая в Европе локальная сеть с 4-мя центрами обработки данных, которая реализует уникальные проекты федерального масштаба с интересными задачами.\nОбязанности:\n• \nРешение инцидентов, обработка запросов на обслуживание, работа с проблемами, рисками, изменениями;\n• \nОперативное реагирование и решение нештатных ситуаций;\n• \nЭксплуатация средств защиты информации (СЗИ);\n• \nУчастие в процессе управления доступом;\n• \nМониторинг несанкционированного доступа;\n• \nМониторинг вирусной активности;\n• \nМониторинг состояния критически важных компонентов средств обеспечения безопасности информации (СОБИ);\n• \nАнализ организации правил сетевых взаимодействий в составе политик сетевой безопасности;\n• \nПроверка данных на наличие конфиденциал

'ФФКУ "Налог-Сервис" ФНС России по ЦОД- это продвинутая IT - организация с новейшим оборудованием и современными технологиями. Мы занимаемся сопровождением IT - инфраструктуры центров обработки данных (ЦОД) налоговой службы.\nНаша организация - крупнейшая в Европе локальная сеть с 4-мя центрами обработки данных, которая реализует уникальные проекты федерального масштаба с интересными задачами.\nОбязанности:\n• \nРешение инцидентов, обработка запросов на обслуживание, работа с проблемами, рисками, изменениями;\n• \nОперативное реагирование и решение нештатных ситуаций;\n•......

In [172]:
df_api_res['vacancyRichText'][0]

'<p>ФФКУ "Налог-Сервис" ФНС России по ЦОД- это продвинутая IT - организация с новейшим оборудованием и современными технологиями. Мы занимаемся сопровождением IT - инфраструктуры центров обработки данных (ЦОД) налоговой службы.<br />Наша организация - крупнейшая в Европе локальная сеть с 4-мя центрами обработки данных, которая реализует уникальные проекты федерального масштаба с интересными задачами.</p><p>Обязанности:</p><ul><li><p>Решение инцидентов, обработка запросов на обслуживание, работа с проблемами, рисками, изменениями;</p></li><li><p>Оперативное реагирование и решение нештатных ситуаций;</p></li><li><p>Эксплуатация средств защиты информации (СЗИ);</p></li><li><p>Участие в процессе управления доступом;</p></li><li><p>Мониторинг несанкционированного доступа;</p></li><li><p>Мониторинг вирусной активности;</p></li><li><p>Мониторинг состояния критически важных компонентов средств обеспечения безопасности информации (СОБИ);</p></li><li><p>Анализ организации правил сетевых взаимоде

<p>ФФКУ "Налог-Сервис" ФНС России по ЦОД- это продвинутая IT - организация с новейшим оборудованием и современными технологиями. Мы занимаемся сопровождением IT - инфраструктуры центров обработки данных (ЦОД) налоговой службы.<br />Наша организация - крупнейшая в Европе локальная сеть с 4-мя центрами обработки данных, которая реализует уникальные проекты федерального масштаба с интересными задачами.</p><p>Обязанности:</p><ul><li><p>Решение инцидентов, обработка запросов на обслуживание, работа с проблемами, рисками, изменениями;</p></li><li><p>Оперативное реагирование и решение нештатных ситуаций;.........

In [173]:
df_api_res = df_api_res.drop(columns=['candidat', 'firm_activity'])

Следующим этапом проверим пропуски. Начнем с более очевидный вариантов - nan

In [174]:
df_api_res.isna().sum().sort_values(ascending=False).head(6)

work                  505
compensation          505
client_staff_count     13
region__title           1
payment_from            0
experience              0
dtype: int64

По результатам первичной проверки на пустые значения 2 признака оказались полностью без данных: work и compensation. По документации оба признака опциональны, но углубившись в данные, можно найти, что "обязанности" = "work" прописаны в признаке 'candidat' вместе с описанием компании, так же как и условия работы, поэтому эти 2 столбца удалим. Возможность опционального заполнения привела к тому, что работодатели все необходимое прописывают в общем описании вакансии, не выделяя отдельно в специальные места.

In [175]:
df_api_res = df_api_res.drop(columns=['work', 'compensation'])

Теперь посмотрим 13 пропусков в client_staff_count:

In [176]:
df_api_res.loc[df_api_res['client_staff_count'].isna(), ['firm_name', 'client_staff_count']]

,firm_name,client_staff_count
50,Тануки,None
88,Тануки,None
121,Тануки,None
163,Тануки,None
175,АО МГЭС,None
198,Тануки,None
230,ТД ПИК,None
231,Тануки,None
298,"ООО ""ИСИНТЕЛ ТЕХНОЛОГИИ""",None
330,"ООО ""ИСИНТЕЛ ТЕХНОЛОГИИ""",None


Пропусков немного, поэтому их можно обработать вручную, чтобы получить более точные значения

In [177]:
df_api_res.loc[df_api_res['firm_name'] == 'ТД ПИК', ['client_staff_count']] = 'менее 50'
df_api_res.loc[df_api_res['firm_name'] == 'Тануки', ['client_staff_count']] = '1000 — 5000'
df_api_res.loc[df_api_res['firm_name'] == 'АО МГЭС', ['client_staff_count']] = '50 — 100'
df_api_res.loc[df_api_res['firm_name'] == 'Обслуживание оборудования.', [ 'client_staff_count']] = '50 — 100'
df_api_res.loc[df_api_res['firm_name'] == 'ООО "ИСИНТЕЛ ТЕХНОЛОГИИ"', ['client_staff_count']] = 'менее 50'
df_api_res.loc[df_api_res['firm_name'] == 'ООО "ММПЗМ"', ['client_staff_count']] = '50 — 100'
df_api_res.loc[df_api_res['firm_name'] == 'Маркетинг и продажи', [ 'client_staff_count']] = '1000 — 5000'
df_api_res.loc[499, 'firm_name'] = 'Дом Моды HENDERSON'

И проверим вакансию, у которой не указан регион:

In [178]:
df_api_res[df_api_res['region__title'].isna()]

,payment_from,payment_to,profession,currency,vacancyRichText,type_of_work,place_of_work,education,experience,firm_name,town_name,region__title,country_title,client_staff_count,position
264,0,1600000,Senior Kotlin разработчик,kzt,"<p>Привет! Мы — <b>Антара</b>, аккредитованная...",Полный рабочий день,Не имеет значения,Не имеет значения,От 3 лет,Антара,Алматы,NaN,Казахстан,100 — 500,"[Разработка, программирование]"


Алматы - город республиканского значения, который не относится к какой-либо области Казахстана. Именно поэтому значение пустое, для изм=бегания возможных будущих ошибок, NaN наменим на пустую строку ''.

In [179]:
df_api_res.loc[df_api_res['region__title'].isna(),'region__title'] = ''

Проверим остальные признаки на пропуски

In [206]:
df_api_res['salary'].value_counts()

salary
0.0         136
200000.0     48
52500.0      15
80000.0      13
100000.0     12
           ... 
520000.0      1
42500.0       1
40800.0       1
86800.0       1
39500.0       1
Name: count, Length: 136, dtype: int64

В некоторых вакансиях зарплаты не указаны, скорее всего это относится к тем, где оплата определяется по договоренности на собеседовании.

In [182]:
df_api_res['profession'].value_counts()[:2]

profession
Ведущий инженер КИПиА      40
Системный администратор    31
Name: count, dtype: int64

In [183]:
df_api_res['currency'].value_counts()[:2]

currency
rub    504
kzt      1
Name: count, dtype: int64

In [184]:
df_api_res['vacancyRichText'].value_counts()

vacancyRichText
<p><b>Вахта в Иркутской области</b></p><p>Мы — динамично развивающаяся компания, специализирующаяся на\nстроительстве и монтаже промышленных объектов. В связи с расширением штата\nприглашаем в команду квалифицированных специалистов для работы на крупных инфраструктурных\nпроектах.<br /><br /> Обязанности</p><p><br />\n- Подготовка технической документации (проекты, сметы, спецификации);</p><p>- Контроль соблюдения сроков и объемов выполненных работ;</p><p>- Организация взаимодействия с подрядчиками и поставщиками\nматериалов;</p><p>- Проведение анализа проектных решений и подготовка предложений по\nоптимизации процессов;</p><p>- Ведение общего журнала работ и специализированных журналов;</p><p>- Обеспечение соответствия выполняемых работ требованиям\nнормативных документов и стандартов</p><p>- Сдача объекта в эксплуатацию</p><p> Требования:</p><p>- Высшее техническое образование </p><p>-\nОпыт работы инженером ПТО в области КИПиА, Электрика (приветствуется) от 3 лет;</p

In [185]:
df_api_res['type_of_work'].value_counts()

type_of_work
Полный рабочий день                       454
Сменный график работы                      20
Частичная занятость / Совместительство     14
Работа вахтовым методом                    13
Неполный рабочий день                       3
Неполная дистанционная занятость            1
Name: count, dtype: int64

Получили, что 90% значений одинаковые - 'Полный рабочий день', для дальнейшего анализа это использовать не будем, так как большинство курсов и так нацелены на процессии с 40 часовой неделей.

In [186]:
df_api_res = df_api_res.drop(columns='type_of_work')

In [187]:
df_api_res['place_of_work'].value_counts()

place_of_work
Не имеет значения             500
Удалённая работа (на дому)      5
Name: count, dtype: int64

'place_of_work' - признак, который можно удалить, так как более 95% значений - 'Не имеет значения', что можно приравнять к константе, поэтому будет мало пользы от анализа этих значений.

In [188]:
df_api_res = df_api_res.drop(columns='place_of_work')

In [189]:
df_api_res['education'].value_counts()

education
Не имеет значения      371
Высшее                  76
Среднее специальное     44
Неполное высшее         13
Среднее                  1
Name: count, dtype: int64

In [190]:
df_api_res['experience'].value_counts()

experience
От 1 года            199
От 3 лет             144
Без опыта            134
Не имеет значения     25
От 6 лет               3
Name: count, dtype: int64

In [191]:
df_api_res['firm_name'].value_counts()[:10]

firm_name
ПСМ                                                         40
Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г.Москве        36
Магнит, Розничная сеть                                      21
Социальный фонд России                                      17
Т1                                                          13
Черногорская ГРК                                            11
Пятёрочка                                                   11
ГК Русагро                                                  10
ФГБОУ ВО Санкт-Петербургский государственный университет     9
Почта России                                                 8
Name: count, dtype: int64

Почистим от мусора тектовый столбец с описаниями вакансий с помощью регулярных выражений.

In [192]:
import re
def clean_diff(diff):
    diff = re.sub(r'\<[^\<\>]+\>', ' ', diff)
    diff = re.sub(r'&nbsp;', ' ', diff)
    diff = re.sub(r'\xa0', ' ', diff)
    diff = re.sub(r'\s\s+', ' ', diff)
    diff = re.sub(r'^[\+\-] ', '', diff)
    
    return diff

df_api_res['vacancyRichText'] = df_api_res['vacancyRichText'].apply(clean_diff)

(Источник, откуда взята функция для очистки данных: https://ru.stackoverflow.com/questions/1349716/Как-очистить-от-тегов-спарсенную-страницу)

In [202]:
df_api_res.head(5)

,profession,currency,vacancyRichText,education,experience,firm_name,town_name,region__title,country_title,client_staff_count,position,salary
0,Дежурный специалист по информационной безопасн...,rub,"ФФКУ ""Налог-Сервис"" ФНС России по ЦОД- это пр...",Не имеет значения,От 1 года,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,Уфа,Республика Башкортостан,Россия,100 — 500,"[Другое, Информационная безопасность]",0.0
1,Системный администратор,rub,"ФКУ ""Налог Сервис"" - это IT-компания, подведо...",Не имеет значения,От 1 года,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,Фурманов,Ивановская область,Россия,100 — 500,"[Администрирование баз данных, Системное админ...",36540.0
2,Системный администратор,rub,ФКУ Налог-Сервис ФНС России - это продвинутая...,Не имеет значения,От 1 года,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,Скадовск,Херсонская область,Россия,100 — 500,"[Администрирование баз данных, Сетевые техноло...",55000.0
3,Системный администратор,rub,"ФКУ ""Налог Сервис"" - это IT-компания, подведо...",Не имеет значения,Без опыта,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,Тосно,Ленинградская область,Россия,100 — 500,"[Администрирование баз данных, Сетевые техноло...",43500.0
4,Системный администратор,rub,"ФКУ ""Налог Сервис"" - это IT-компания, подведо...",Не имеет значения,Без опыта,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,Скадовск,Херсонская область,Россия,100 — 500,"[Администрирование баз данных, Сетевые техноло...",32500.0


Соберем в один датасет названия города, региона и страны, чтобы дальше можно было удобнее обогатить данные с хх.ру

In [219]:
towns_region = to_csv_towns.merge(df_regions, on='id_region',  how='left')
towns_regions_contries = towns_region.merge(df_countries, on='id_country',how='left')

In [221]:
towns_regions_contries = towns_regions_contries.drop(columns=['id',	'id_region','id_country', 'title_eng'])

In [224]:
towns_regions_contries.head(5)

,title,region__title,country_title
0,Москва,Московская область,Россия
1,Санкт-Петербург,Ленинградская область,Россия
2,Новосибирск,Новосибирская область,Россия
3,Екатеринбург,Свердловская область,Россия
4,Нижний Новгород,Нижегородская область,Россия


In [225]:
towns_regions_contries.to_csv('./towns_regions_contries.csv', index=False)

In [231]:
company_staff = df_api_res.loc[:, ['firm_name', 'client_staff_count']]
company_staff= company_staff.drop_duplicates()

In [232]:
company_staff.to_csv('./company_staff.csv', index=False)

Выгрузим получившийся датасет

In [237]:
df_api_res.to_csv('../analysis/df_sj_api.csv', index=False)